#### Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import KMRF class
from kmrf import KMRF
from KMRF_training_config import *

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  Pandas version: 2.2.1
  NumPy version: 1.26.4


## 1. Configure KMRF Model

In [3]:
# Create DataFrame of asset names
asset_names_df = pd.DataFrame({
    'universe': get_assets_by_class('universe') + ['']*7,
    'us_equity': get_assets_by_class('us_equity'),
    'commodity': get_assets_by_class('commodity') + ['']*5,
    'int_equity': get_assets_by_class('int_equity') + ['']*11,

})

from pandas import option_context
with option_context('display.max_colwidth', None):
    display(asset_names_df)

,universe,us_equity,commodity,int_equity
0,IVV - iShares Core S&P 500 ETF,SPDR S&P 500 ETF,Gold Futures,Vanguard Total International Stock ETF
1,IJH - iShares Core S&P Mid-Cap ETF,Invesco QQQ Trust,Wheat Futures,Vanguard FTSE Developed Markets ETF
2,IWM - iShares Russell 2000 ETF,iShares Russell 2000 ETF,Corn Futures,Vanguard FTSE Emerging Markets ETF
3,EFA - iShares MSCI EAFE ETF,SPDR Dow Jones Industrial Average ETF,Copper,Vanguard FTSE Europe ETF
4,EEM - iShares MSCI Emerging Markets ETF,Energy Select Sector SPDR,Sugar,Vanguard FTSE Pacific ETF
5,AGG - iShares Core U.S. Aggregate Bond ETF,Financial Select Sector SPDR,Silver Futures,iShares China Large-Cap ETF
6,SPTL - SPDR Portfolio Long Term Treasury ETF,Utilities Select Sector SPDR,US Dollar,iShares MSCI Japan ETF
7,HYG - iShares iBoxx $ High Yield Corporate Bond ETF,Industrial Select Sector SPDR,Soybean Futures,iShares MSCI India ETF
8,SPBO - SPDR Portfolio Corporate Bond ETF,Health Care Select Sector SPDR,Lumber Futures,
9,IYR - iShares U.S. Real Estate ETF,Technology Select Sector SPDR,Live Cattle Futures,


## 2. Batch Train KMRF Model

In [4]:
for asset_class in ['us_equity']:
    for asset_name in asset_names_df[asset_class].tolist():
        TRAINING_CONFIG = KMRF_Training_Config(
            asset_name= asset_name,
            classification_type='original',
            use_data_type='master',
            end_date='20181231',
            feature_window_size=1,
            feature_asset_classes=[],
            cross_asset_specific=[],  # empty means all (only referring to universe assets)
            use_boruta_selection=True,
            use_consensus_selection=False
        )

        model = KMRF(
            asset_class=TRAINING_CONFIG.get_asset_class(),
            asset_name=TRAINING_CONFIG.get_asset_name(),
            classification_type=TRAINING_CONFIG.get_classification_type(),
            end_date=TRAINING_CONFIG.get_date_ranges()['end_date'],
            use_data_type=TRAINING_CONFIG.get_use_data_type(),
            feature_window_size=TRAINING_CONFIG.get_feature_window_size(),  
            feature_asset_classes=TRAINING_CONFIG.get_cross_asset_features(),
            cross_asset_specific=TRAINING_CONFIG.get_cross_asset_specific(),
            xgb_params=TRAINING_CONFIG.get_xgb_params(),
            use_boruta_selection=TRAINING_CONFIG.get_use_boruta_selection(),
            use_consensus_selection=TRAINING_CONFIG.get_use_consensus_selection(),
        )

        model.pipeline(optimize=False)

        path = f'saved_models/KMRF_new/{model.classification_type}/{model.asset_class}/'
        path += f'{model.asset_name.replace(" ", "_")}_KMRF_model.pkl'
        model.save_model(path)

KMRF model initialized
  Asset: SPDR S&P 500 ETF
  Asset class: us_equity
  Classification type: original
  End date: 20181231
  Data type: master
  Data path: C:\Users\jesse\Trivariate Dropbox\RESEARCH\workspace\jesse\playground\FE800_project_code\data\master_df.csv
  KAMA+MSR model directory: C:\Users\jesse\Trivariate Dropbox\RESEARCH\workspace\jesse\playground\FE800_project_code\saved_models\KAMA_MSR\us_equity\20181231
  Validation period: 2019-01-02 to 2021-12-31
  Test start: 2022-01-02
  Random seed: 1010
  Feature window size: 1 days
  Feature asset classes: []
  Feature selection: Boruta=True, Consensus=False
  Custom XGB parameters: {'n_estimators': 220, 'max_depth': 13, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.25, 'min_child_weight': 95, 'gamma': 0.045, 'random_state': 1010, 'n_jobs': -1, 'tree_method': 'hist', 'enable_categorical': False}

KMRF PIPELINE FOR SPDR S&P 500 ETF
Asset Class: us_equity
Classification Type: original
Data Type: master
Feature As

KeyboardInterrupt: 